In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
import sys
import seaborn as sns

from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.utils.constants import HELVETICA_LIGHT_PATH, HELVETICA_PATH

In [ ]:
def compute_bias(df, sample_var_col, pop_var_col):
    # compute bias for a dataframe of observed and expected values
    # returns a pd.Series
    observed_proportion = df[sample_var_col] / df[sample_var_col].sum() * 100
    expected_proportion = df[pop_var_col] / df[pop_var_col].sum() * 100
    bias = observed_proportion - expected_proportion
    return bias


def make_center_line_coords(df, x, y):
    # This creates two coords to define the hypothetical line in which x and y are perfectly correlatd
    expected_slope = df[y].sum() / df[x].sum()
    xcoord = np.array([df[x].min(), df[x].max()])
    ycoord = expected_slope * xcoord
    return (xcoord, ycoord)


def summarize_bias(df, bias_col):
    sum_abs_bias = np.absolute(df[bias_col]).sum()
    print("Sum Absolute Bias: {0:.2f}".format(sum_abs_bias))
    return None


def plot_claims_vs_census(
    df,
    xvar,
    yvar,
    bias_col="bias",
    xlim_=None,
    ylim_=None,
    text_col=None,
    num_to_label=5,
    alpha=0.9,
    labelfontsize=14,
    filepath="census_bias.pdf",
):
    # `text_col` is the column used to annotate the points
    # `num_to_label` sets the top num_to_label records sorted by yvar to be annotated

    helvetica_properties = font_manager.FontProperties(fname=HELVETICA_PATH)
    helvetica_light_properties = font_manager.FontProperties(fname=HELVETICA_LIGHT_PATH)

    plt.rcParams["figure.figsize"] = [7, 7]
    sns.set_style("whitegrid")

    # Report Stats
    pearson_r = df[[xvar, yvar]].corr().loc[xvar, yvar]
    print("pearson r: {0:.3f}".format(pearson_r))
    summarize_bias(df, bias_col)

    # scatter plot
    ax = df.plot.scatter(
        xvar, yvar, alpha=alpha, xlim=xlim_, ylim=ylim_, color="tab:purple", s=70
    )
    ax.set_xlabel(xvar, fontproperties=helvetica_properties)
    ax.set_ylabel(yvar, fontproperties=helvetica_properties)
    ax.set_xticklabels(
        [int(element) for element in ax.get_xticks()],
        fontproperties=helvetica_light_properties,
    )
    ax.set_yticklabels(
        [int(element) for element in ax.get_yticks()],
        fontproperties=helvetica_light_properties,
    )

    # annotate
    xadj = df[xvar].max() * 0.02
    if text_col is not None:
        for i, point in (
            df.sort_values(by=xvar, ascending=False).iloc[0:num_to_label].iterrows()
        ):
            ax.text(
                point[xvar] + xadj,
                point[yvar],
                str(point[text_col]),
                fontsize=labelfontsize,
                verticalalignment="center",
            )
    # plot expectation line
    line_coords = make_center_line_coords(df, xvar, yvar)
    plt.plot(line_coords[0], line_coords[1], linestyle="-", linewidth=4, color="indigo")

    ax.set_title(
        "Population Size by ZIP3: Alignment of Komodo and Census Data",
        fontproperties=helvetica_properties,
        fontsize=12,
    )
    ax.text(
        0.05,
        0.95,
        f"Pearson r = {pearson_r:.3f}",
        transform=ax.transAxes,
        fontsize=12,
        verticalalignment="top",
        bbox=dict(facecolor="white", alpha=0.5),
        fontproperties=helvetica_light_properties,
    )

    plt.savefig(filepath, format="pdf", dpi=300, bbox_inches="tight")
    print(f"SAVED figure to {filepath}")
    return ax

In [ ]:
from src.utils.constants import CENSUS_DATA_PATH

census_data_path = CENSUS_DATA_PATH
race_filename = "DECENNIALDHC2020.P9-Data.csv"
education_filename = "ACSST5Y2020.S1501-Data.csv"
income_filename = "ACSST5Y2020.S1901-Data.csv"
population_filename = "DECENNIALDHC2020.P1-Data.csv"
age_sex_filename = "ACSST5Y2020.S0101-Data.csv"


def extract_zip5(area_name):
    return area_name.split(" ")[1]


def get_processed_census_csv(filepath):
    df = pd.read_csv(filepath)
    new_header = df.iloc[0]
    df = df[1:]
    df.columns = new_header

    df["ZIP_5"] = df["Geographic Area Name"].apply(
        lambda area_name: extract_zip5(area_name)
    )
    assert len(df["ZIP_5"].unique()) == len(df["ZIP_5"])

    df["ZIP3"] = df["ZIP_5"].str[:3]
    num_unique = df["ZIP3"].nunique()
    print(f"{num_unique} unique ZIP3s")
    return df


df_census = get_processed_census_csv(
    os.path.join(census_data_path, population_filename)
)
df_census["Total Population"] = df_census[" !!Total"].astype(int)
df_census_grouped = df_census.groupby("ZIP3")[["Total Population"]].sum().reset_index()

In [ ]:
dfs = []

from src.utils.constants import DATA_PATH

geography_dir = str(Path(DATA_PATH) / "komodo_geographic")
for filename in os.listdir(geography_dir):
    if filename.endswith(".csv"):
        dfs.append(
            pd.read_csv(
                os.path.join(geography_dir, filename),
                usecols=["PATIENT_ID", "PATIENT_ZIP3"],
            )
        )

df_komodo = pd.concat(dfs)
df_komodo.drop_duplicates(subset=["PATIENT_ID"], ignore_index=True, inplace=True)

df_komodo["ZIP_3_str"] = df_komodo["PATIENT_ZIP3"].astype(str)
df_komodo["ZIP_3_str"] = df_komodo["ZIP_3_str"].str.zfill(3)
df_komodo_grouped = df_komodo.groupby("ZIP_3_str").count().reset_index()
df_komodo_grouped["Total Population"] = df_komodo_grouped["PATIENT_ID"]
df_komodo_grouped.drop(columns=["PATIENT_ID", "PATIENT_ZIP3"], inplace=True)

In [ ]:
df_joined = df_komodo_grouped.merge(
    df_census_grouped,
    left_on="ZIP_3_str",
    right_on="ZIP3",
    how="outer",
    suffixes=("_komodo", "_census"),
)
df_joined["Total Population_komodo"] = df_joined["Total Population_komodo"].fillna(0)
df_joined["Total Population_census"] = df_joined["Total Population_census"].fillna(0)

In [ ]:
df_joined["bias"] = compute_bias(
    df_joined,
    sample_var_col="Total Population_komodo",
    pop_var_col="Total Population_census",
)

In [ ]:
xvar = "Komodo ZIP3 Population"
yvar = "Census ZIP3 Population"
df_joined[xvar] = df_joined["Total Population_komodo"]
df_joined[yvar] = df_joined["Total Population_census"]

ax = plot_claims_vs_census(
    df_joined, xvar, yvar, text_col=None, num_to_label=0, alpha=0.5
)